# Agrupamento de Dados Sísmicos e Generalização entre Bacias Sedimentares

## Análise Comparativa dos Algoritmos K-means e Gaussian Mixture Model (GMM)

### 1. Introdução

A observação de dados sísmicos é uma das ferramentas mais importante para a caracterização de estruturas geológicas em subsurperfícies. Porém, diante dos dados trabalhados, existe o problema dos elevandos volumes de informação e a complexidade dos sinais, tornando o processo de interpretação algo bastante desafiador, motivando o uso de técnicas de Machine Learning para ajudar no processo de identificação de padrões.

Neste trabalho, são investigados dois algoritmos de aprendizado não supervisionado: K-means e Gaussian Mixture Model (GMM). A aplicação desses modelos será realizada a partir da extração de atributos sísmicos, permitindo agrupar amostras com características semelhantes e analisar sua distribuição nos dados.

Primeiramente, serão usados dados sísmicos pré-stack da Bacia de Jequitinhonha para o processo de desenvolvimento, treinamento e validação interna dos modelos. Em seguida, os modelos serão aplicados a dados de outra bacia sedimentar, a Bacia de Campos, com a finalidade de avaliar sua capacidade de generalização em diferentes contextos geológicos.

Dentre os aspectos analisados, serão aplicados a qualidade dos agrupamentos, otimização de hiperparâmetros incerteza das classificações probabilísticas e ao desempenho computacional dos algoritmos.

### 2. Importação das Bibliotecas e Configurações

Nesta etapa, são importadas as bibliotecas necessárias para
o processamento dos dados sísmicos, a extração de atributos,
a implementação dos modelos de Machine Learning e a
visualização dos resultados.

In [ ]:
# Manipulação de arquivos e sistema
import os
import sys
from pathlib import Path
import time

# Processamento numérico
import numpy as np
import pandas as pd

# Processamento de sinais sísmicos
import segyio
from scipy.signal import hilbert
from scipy.ndimage import uniform_filter1d

# Machine Learning
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Métricas de avaliação
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score
)

# Visualização
import matplotlib.pyplot as plt

# Armazenamento de modelos
import joblib

# Monitoramento computacional
import psutil

print("Bibliotecas importadas com sucesso!")

#### 2.1 Configuração dos parâmetros gerais

In [ ]:
# Reprodutibilidade
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# Separação dos dados
TRAIN_SIZE = 0.64
TUNING_SIZE = 0.16
VALIDATION_SIZE = 0.20

# Limites de amostragem
MAX_TRAIN_SAMPLES = 100_000
MAX_VALIDATION_SAMPLES = 20_000
SILHOUETTE_SAMPLES = 5_000

# Intervalo de clusters avaliados
K_RANGE = range(2, 10)

# Processamento em blocos
BATCH_SIZE = 100_000
TRACE_BATCH_SIZE = 128

# Limites para visualização
MAX_DISPLAY_TRACES = 800
MAX_DISPLAY_SAMPLES = 1500

# Configuração dos gráficos
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "figure.dpi": 100,
    "font.size": 11
})


### 3. Carregamento dos Dados Sísmicos

In [ ]:
# Pasta principal do projeto
PROJECT_ROOT = Path.cwd().parent

# Diretórios do projeto
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"
METRICS_DIR = PROJECT_ROOT / "results" / "metrics"

# Permitir a importação dos módulos da pasta src
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
# Dados de treinamento
JEQUITINHONHA_PATH = DATA_RAW / "jequitinhonha.sgy"

# Dados de teste externo
CAMPOS_PATH = DATA_RAW / "campos.sgy"

# Verificação dos arquivos
print("Jequitinhonha:", JEQUITINHONHA_PATH.is_file())
print("Campos:", CAMPOS_PATH.is_file())

#### 3.1. Carregamento dos dados da Bacia do Jequitinhonha

In [ ]:
from src.data_loader import load_segy_manual_2d_final

seismic_data, inlines, crosslines, times = (
    load_segy_manual_2d_final(JEQUITINHONHA_PATH)
)

print("Dados do Jequitinhonha carregados!")
print("Dimensões:", seismic_data.shape)
print("Número de traços:", len(inlines))
print("Amostras por traço:", len(times))

#### 3.2. Carregamento dos dados da Bacia de Campos

In [ ]:
(
    seismic_data_campos,
    inlines_campos,
    crosslines_campos,
    times_campos
) = load_segy_manual_2d_final(CAMPOS_PATH)

print("Dados de Campos carregados!")
print("Dimensões:", seismic_data_campos.shape)
print("Número de traços:", len(inlines_campos))
print("Amostras por traço:", len(times_campos))

#### 3.3. Comparação das características dos dados sísmicos

In [ ]:
with segyio.open(
    str(JEQUITINHONHA_PATH), "r", ignore_geometry=True
) as f:
    dt_jequitinhonha = f.bin[segyio.BinField.Interval] / 1000

with segyio.open(
    str(CAMPOS_PATH), "r", ignore_geometry=True
) as f:
    dt_campos = f.bin[segyio.BinField.Interval] / 1000


df_metadata = pd.DataFrame({
    "Jequitinhonha": {
        "Número de traços": seismic_data.shape[0],
        "Amostras por traço": seismic_data.shape[2],
        "Total de amostras": seismic_data.size,
        "Intervalo de amostragem (ms)": dt_jequitinhonha,
        "Memória utilizada (MiB)": round(
            seismic_data.nbytes / 1024**2, 2
        )
    },
    "Campos": {
        "Número de traços": seismic_data_campos.shape[0],
        "Amostras por traço": seismic_data_campos.shape[2],
        "Total de amostras": seismic_data_campos.size,
        "Intervalo de amostragem (ms)": dt_campos,
        "Memória utilizada (MiB)": round(
            seismic_data_campos.nbytes / 1024**2, 2
        )
    }
})

display(df_metadata)

### 4. Análise Exploratória dos Dados

A análise exploratória tem como objetivo investigar as características dos sinais sísmicos das bacias do Jequitinhonha e de Campos.

#### 4.1 Caracterização estatística

In [ ]:
def seismic_statistics(data):

    valid_mask = np.isfinite(data)
    valid_data = data[valid_mask]

    if valid_data.size == 0:
        raise ValueError("Nenhuma amostra válida.")

    return {
        "Amplitude mínima": np.min(valid_data),
        "Amplitude máxima": np.max(valid_data),
        "Média": np.mean(valid_data, dtype=np.float64),
        "Desvio padrão": np.std(
            valid_data, dtype=np.float64
        ),
        "Mediana": np.median(valid_data),
        "Percentil 99,5 absoluto": np.percentile(
            np.abs(valid_data), 99.5
        ),
        "Valores inválidos (%)": (
            100 * (1 - valid_mask.mean())
        ),
        "Valores zerados (%)": (
            100 * np.mean(valid_data == 0)
        )
    }


df_statistics = pd.DataFrame({
    "Jequitinhonha": seismic_statistics(seismic_data),
    "Campos": seismic_statistics(seismic_data_campos)
})

display(df_statistics.round(4))

#### 4.2. Visualização da amplitude sísmica

Os dados sísmicos são reorganizados em matrizes bidimensionais para visualização das amplitudes.

In [ ]:
# Conversão para matrizes bidimensionais
seismic_data_2d = seismic_data.reshape(
    -1, seismic_data.shape[2]
)

seismic_data_campos_2d = seismic_data_campos.reshape(
    -1, seismic_data_campos.shape[2]
)


def plot_seismic_amplitude(data, titulo):

    # Ajuste do contraste
    v_limit = np.nanpercentile(np.abs(data), 99.5)
    v_limit = v_limit if v_limit > 0 else 1.0

    # Redução de resolução apenas para visualização
    step = max(1, int(np.ceil(data.shape[0] / 1200)))

    data_display = data[::step]

    fig, ax = plt.subplots(figsize=(12, 6))

    im = ax.imshow(
        data_display.T,
        aspect="auto",
        cmap="seismic",
        vmin=-v_limit,
        vmax=v_limit,
        extent=[
            0,
            data.shape[0],
            data.shape[1],
            0
        ]
    )

    ax.set_title(titulo)
    ax.set_xlabel("Índice do traço")
    ax.set_ylabel("Amostras temporais")

    plt.colorbar(
        im,
        ax=ax,
        orientation="horizontal",
        fraction=0.046,
        pad=0.08,
        label="Amplitude sísmica"
    )

    plt.tight_layout()
    plt.show()


plot_seismic_amplitude(
    seismic_data_2d,
    "Amplitude Sísmica — Jequitinhonha"
)

plot_seismic_amplitude(
    seismic_data_campos_2d,
    "Amplitude Sísmica — Campos"
)

#### 4.3. Distribuição das amplitudes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

datasets = {
    "Jequitinhonha": seismic_data,
    "Campos": seismic_data_campos
}

for ax, (name, data) in zip(axes, datasets.items()):

    values = data.ravel()
    values = values[np.isfinite(values)]

    local_rng = np.random.default_rng(RANDOM_STATE)

    indices = local_rng.choice(
        len(values),
        size=min(100_000, len(values)),
        replace=False
    )

    sample = values[indices]

    ax.hist(
        sample,
        bins=100,
        color="steelblue",
        density=True,
        edgecolor="black",
        alpha=0.8
    )

    ax.set_title(name)
    ax.set_xlabel("Amplitude sísmica")
    ax.set_ylabel("Densidade")
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

### 4.4. Análise espectral dos sinais

O conteúdo de frequência dos sinais sísmicos é investigado utilizando o método de Welch.

In [ ]:
from scipy.signal import welch


def mean_frequency_spectrum(data, dt_ms):

    data = data.reshape(-1, data.shape[-1])

    # Seleção de traços distribuídos pelo arquivo
    indices = np.linspace(
        0,
        len(data) - 1,
        min(256, len(data)),
        dtype=int
    )

    sample = data[indices]

    valid = (
        np.isfinite(sample).all(axis=1)
        & np.any(sample != 0, axis=1)
    )

    sample = sample[valid]

    fs = 1000 / dt_ms

    frequencies, power = welch(
        sample,
        fs=fs,
        axis=1,
        nperseg=min(256, sample.shape[1])
    )

    mean_power = power.mean(axis=0)

    return frequencies, mean_power / max(
        mean_power.max(), 1e-12
    )


freq_j, power_j = mean_frequency_spectrum(
    seismic_data, dt_jequitinhonha
)

freq_c, power_c = mean_frequency_spectrum(
    seismic_data_campos, dt_campos
)

plt.figure(figsize=(10, 5))

plt.plot(freq_j, power_j, label="Jequitinhonha")
plt.plot(freq_c, power_c, label="Campos")

plt.title("Comparação dos Espectros Médios")
plt.xlabel("Frequência (Hz)")
plt.ylabel("Potência normalizada")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 5. Extração de Atributos Sísmicos

A engenharia de atributos consiste na transformação das amplitudes sísmicas em um conjunto de características utilizadas como entrada dos algoritmos ML.

#### 5.1. Transformada de Hilbert e sinal analítico

O sinal analítico é obtido pela combinação do sinal
sísmico original com sua transformada de Hilbert:

$$
S_a(t)=S(t)+jH[S(t)]
$$

#### Envelope

$$
E(t)=|S_a(t)|
$$

#### Fase instantânea

$$
\phi(t)=\operatorname{atan2}
\left(
\operatorname{Im}(S_a(t)),
\operatorname{Re}(S_a(t))
\right)
$$

#### Frequência instantânea

$$
f(t)=\frac{1}{2\pi}\frac{d\phi(t)}{dt}
$$

#### Amplitude RMS

$$
RMS=\sqrt{\frac{1}{N}\sum_{i=1}^{N}S_i^2}
$$

#### Variância local

$$
\sigma^2=\frac{1}{N}\sum_{i=1}^{N}(S_i-\mu)^2
$$

Os atributos são calculados ao longo do eixo temporal
de cada traço sísmico.

#### 5.2. Implementação da extração de atributos

A transformada de Hilbert é aplicada ao eixo temporal
dos dados sísmicos, preservando a estrutura matricial
utilizada na metodologia original.

A extração é realizada inicialmente em blocos reduzidos
de traços, visando limitar o consumo de memória.

In [ ]:
import importlib
import src.features

importlib.reload(src.features)

from src.features import (
    extract_seismic_attributes,
    create_feature_matrix
)

print("Funções importadas com sucesso!")

In [ ]:
from src.features import (
    extract_seismic_attributes,
    create_feature_matrix
)

# Selecionar blocos centrais para visualização
start_j = max(0, seismic_data.shape[0] // 2 - 200)
start_c = max(0, seismic_data_campos.shape[0] // 2 - 200)

sample_jequitinhonha = seismic_data[start_j:start_j + 400]
sample_campos = seismic_data_campos[start_c:start_c + 400]

# Extração dos atributos
features_jequitinhonha = extract_seismic_attributes(
    sample_jequitinhonha,
    dt_jequitinhonha
)

features_campos = extract_seismic_attributes(
    sample_campos,
    dt_campos
)

print("Atributos extraídos com sucesso!")

for name, values in features_jequitinhonha.items():
    print(name, values.shape)

#### 5.3. Visualização dos atributos sísmicos

In [ ]:
def plot_seismic_attributes(features_list, basin_name):

    fig, axes = plt.subplots(
        2, 3, figsize=(14, 8)
    )

    color_maps = {
        "Amplitude": "seismic",
        "Envelope": "inferno",
        "Fase": "twilight",
        "Frequência": "coolwarm",
        "RMS": "viridis",
        "Variância": "magma"
    }

    for ax, (name, values) in zip(
        axes.flat,
        features_list.items()
    ):

        data = values.reshape(-1, values.shape[-1])

        if name == "Fase":

            vmin, vmax = -np.pi, np.pi

        elif name in ["Amplitude", "Frequência"]:

            limit = np.percentile(
                np.abs(data), 99.5
            ) or 1.0

            vmin, vmax = -limit, limit

        else:

            vmin = 0
            vmax = np.percentile(data, 99.5) or 1.0

        im = ax.imshow(
            data.T,
            aspect="auto",
            cmap=color_maps[name],
            vmin=vmin,
            vmax=vmax
        )

        ax.set_title(name)
        ax.set_xlabel("Traços")
        ax.set_ylabel("Amostras")

        fig.colorbar(im, ax=ax, fraction=0.046)

    fig.suptitle(
        f"Atributos Sísmicos — {basin_name}",
        fontsize=15
    )

    plt.tight_layout()
    plt.show()


plot_seismic_attributes(
    features_jequitinhonha,
    "Jequitinhonha"
)

plot_seismic_attributes(
    features_campos,
    "Campos"
)

### 5.4. Normalização das amplitudes sísmicas

A análise dos atributos revelou diferenças significativas
entre as escalas de amplitude dos dados das bacias do
Jequitinhonha e de Campos.

Para reduzir a influência dessas diferenças sobre os
modelos de Machine Learning, é aplicada uma normalização
robusta das amplitudes utilizando o percentil 99,5%
da amplitude absoluta de cada traço sísmico.

Após a normalização, os atributos são recalculados,
permitindo uma comparação baseada nas características
relativas dos sinais.

In [ ]:
# Atributos normalizados do Jequitinhonha
features_jequitinhonha_normalized = extract_seismic_attributes(
    sample_jequitinhonha,
    dt_jequitinhonha,
    normalize=True
)

# Atributos normalizados de Campos
features_campos_normalized = extract_seismic_attributes(
    sample_campos,
    dt_campos,
    normalize=True
)

print("Atributos normalizados extraídos com sucesso!")

In [ ]:
plot_seismic_attributes(
    features_jequitinhonha_normalized,
    "Jequitinhonha — Normalizado"
)

plot_seismic_attributes(
    features_campos_normalized,
    "Campos — Normalizado"
)

### 6. Separação e Pré-processamento dos Dados

Para a realização do treinamento, se faz necessário uma separação prévia dos dados em conjuntos de treinamento, ajuste de
hiperparâmetros e validação interna.

Os dados da Bacia de Campos são reservados para a avaliação da capacidade de generalização dos modelos.

### 6.1. Separação dos conjuntos de dados

A separação dos dados é realizada por blocos contíguos
de traços sísmicos da Bacia do Jequitinhonha.

Os dados são distribuídos da seguinte maneira:

- 64% para treinamento;
- 16% para ajuste de hiperparâmetros;
- 20% para validação interna.

In [ ]:
# Número total de traços
n_traces = seismic_data.shape[0]
n_traces_campos = seismic_data_campos.shape[0]

# Verificar as proporções
assert np.isclose(
    TRAIN_SIZE + TUNING_SIZE + VALIDATION_SIZE,
    1.0
)

assert min(
    TRAIN_SIZE,
    TUNING_SIZE,
    VALIDATION_SIZE
) > 0

# Limites dos conjuntos
cut_train = int(n_traces * TRAIN_SIZE)

cut_validation = int(
    n_traces * (TRAIN_SIZE + TUNING_SIZE)
)

# Índices dos traços de treinamento
train_indices = np.arange(0, cut_train)

# Índices para ajuste de hiperparâmetros
tuning_indices = np.arange(
    cut_train,
    cut_validation
)

# Índices para validação interna
validation_indices = np.arange(
    cut_validation,
    n_traces
)

# Índices da Bacia de Campos
campos_indices = np.arange(n_traces_campos)

# Resultados
print("Total de traços do Jequitinhonha:", n_traces)

print("Treinamento:", len(train_indices))
print("Ajuste:", len(tuning_indices))
print("Validação:", len(validation_indices))

print("\nTotal de traços de Campos:", n_traces_campos)
print("Teste externo:", len(campos_indices))

In [ ]:
# Verificar se os conjuntos não estão vazios
assert len(train_indices) > 0
assert len(tuning_indices) > 0
assert len(validation_indices) > 0

# Verificar sobreposição de traços
assert np.intersect1d(
    train_indices,
    tuning_indices
).size == 0

assert np.intersect1d(
    train_indices,
    validation_indices
).size == 0

assert np.intersect1d(
    tuning_indices,
    validation_indices
).size == 0

# Verificar se todos os traços foram considerados
all_indices = np.concatenate([
    train_indices,
    tuning_indices,
    validation_indices
])

assert np.array_equal(
    np.sort(all_indices),
    np.arange(n_traces)
)

print("Separação validada com sucesso!")
print("Nenhum traço compartilhado entre os conjuntos.")
print("Todos os traços foram considerados.")

In [ ]:
df_split = pd.DataFrame({

    "Conjunto": [
        "Treinamento",
        "Ajuste de hiperparâmetros",
        "Validação interna",
        "Teste externo"
    ],

    "Bacia": [
        "Jequitinhonha",
        "Jequitinhonha",
        "Jequitinhonha",
        "Campos"
    ],

    "Número de traços": [
        len(train_indices),
        len(tuning_indices),
        len(validation_indices),
        len(campos_indices)
    ],

    "Percentual (%)": [
        100 * len(train_indices) / n_traces,
        100 * len(tuning_indices) / n_traces,
        100 * len(validation_indices) / n_traces,
        100.0
    ]
})

display(df_split.round(2))

#### 6.2. Construção da matriz de atributos sísmicos

Os atributos sísmicos são organizados em uma matriz
bidimensional, na qual cada linha representa uma amostra
sísmica e cada coluna representa uma característica.

In [ ]:
# Atributos da metodologia original
original_feature_names = [
    "Amplitude",
    "Envelope",
    "Fase"
]

# Atributos da metodologia expandida
feature_names = [
    "Amplitude",
    "Envelope",
    "Seno da fase",
    "Cosseno da fase",
    "Frequência",
    "RMS",
    "Variância"
]

# Todos os atributos disponíveis
all_feature_names = feature_names + ["Fase"]

print("Atributos da metodologia original:")
print(original_feature_names)

print("\nAtributos da metodologia expandida:")
print(feature_names)

print("\nTotal de colunas armazenadas:")
print(len(all_feature_names))

In [ ]:
from src.features import (
    extract_seismic_attributes,
    create_feature_matrix,
    normalize_seismic_traces
)

# Pequeno bloco para verificação
test_block = seismic_data[:10].copy()

# Extração sem normalização
test_original = extract_seismic_attributes(
    test_block,
    dt_jequitinhonha,
    normalize=False
)

# Extração com normalização
test_normalized = extract_seismic_attributes(
    test_block,
    dt_jequitinhonha,
    normalize=True
)

# Normalização de referência
expected_normalized = normalize_seismic_traces(
    test_block
)

# Verificar preservação da amplitude original
assert np.array_equal(
    test_original["Amplitude"],
    test_block
)

# Verificar aplicação da normalização
assert np.allclose(
    test_normalized["Amplitude"],
    expected_normalized
)

# Verificar preservação dos dados de entrada
assert np.array_equal(
    test_block,
    seismic_data[:10]
)

print("Normalização verificada com sucesso!")
print("Dados originais preservados.")

#### 6.4. Construção das amostras


In [ ]:
def build_sample(
    data,
    trace_indices,
    dt_ms,
    max_samples,
    seed=42
):

    # Gerador de números aleatórios
    local_rng = np.random.default_rng(seed)

    # Verificar formato dos dados
    if data.ndim != 3 or data.shape[1] != 1:
        raise ValueError(
            "Formato esperado: (n_traces, 1, n_samples)."
        )

    trace_indices = np.asarray(
        trace_indices,
        dtype=np.int64
    )

    n_samples = data.shape[2]

    # Número total de amostras disponíveis
    n_total = len(trace_indices) * n_samples

    if n_total == 0 or max_samples <= 0:
        raise ValueError("Conjunto de dados vazio.")

    # Seleção aleatória sem reposição
    selected = np.sort(
        local_rng.choice(
            n_total,
            size=min(max_samples, n_total),
            replace=False
        )
    )

    # Identificação dos traços e posições temporais
    selected_traces = selected // n_samples
    selected_times = selected % n_samples

    parts = []

    # Processamento em blocos de traços
    for start in range(
        0,
        len(trace_indices),
        TRACE_BATCH_SIZE
    ):

        stop = min(
            start + TRACE_BATCH_SIZE,
            len(trace_indices)
        )

        # Identificar as amostras pertencentes ao bloco
        mask = (
            (selected_traces >= start)
            & (selected_traces < stop)
        )

        if not mask.any():
            continue

        # Selecionar os dados sísmicos originais
        block = data[trace_indices[start:stop]]

        # Extrair atributos COM NORMALIZAÇÃO
        features = extract_seismic_attributes(
            block,
            dt_ms,
            normalize=True
        )

        # Índices relativos ao bloco
        row_indices = selected_traces[mask] - start
        time_indices = selected_times[mask]

        # Selecionar as amostras dos atributos
        sampled_features = {
            name: values[row_indices, 0, time_indices]
            for name, values in features.items()
        }

        # Empilhamento e construção da matriz
        matrix = create_feature_matrix(
            sampled_features,
            all_feature_names
        )

        # Verificar validade das amostras
        valid = (
            np.isfinite(matrix).all(axis=1)
            & (sampled_features["Envelope"] > 0)
        )

        # Armazenar as amostras válidas
        parts.append(matrix[valid])

    if not parts:
        raise ValueError("Nenhuma amostra disponível.")

    # Unir todas as matrizes
    X = np.concatenate(parts, axis=0)

    if len(X) == 0:
        raise ValueError("Nenhuma amostra válida.")

    return X.astype(np.float32, copy=False)

In [ ]:
# Conjunto de treinamento
X_train = build_sample(
    seismic_data,
    train_indices,
    dt_jequitinhonha,
    MAX_TRAIN_SAMPLES,
    seed=RANDOM_STATE
)

print("Treinamento preparado!")

# Conjunto de ajuste de hiperparâmetros
X_tuning = build_sample(
    seismic_data,
    tuning_indices,
    dt_jequitinhonha,
    MAX_VALIDATION_SAMPLES,
    seed=RANDOM_STATE + 1
)

print("Ajuste preparado!")

# Conjunto de validação interna
X_val = build_sample(
    seismic_data,
    validation_indices,
    dt_jequitinhonha,
    MAX_VALIDATION_SAMPLES,
    seed=RANDOM_STATE + 2
)

print("Validação preparada!")

# Dimensões das matrizes
print("\nDimensões das matrizes:")

print("X_train:", X_train.shape)
print("X_tuning:", X_tuning.shape)
print("X_val:", X_val.shape)

#### 6.5. Análise e verificação das matrizes

As matrizes de atributos são analisadas quanto às suas
dimensões, características estatísticas e presença de
valores inválidos.

In [ ]:
df_train = pd.DataFrame(
    X_train,
    columns=all_feature_names
)

display(df_train.head(10))

In [ ]:
df_attributes_statistics = df_train.describe().T

display(df_attributes_statistics.round(4))

In [ ]:
amplitude_index = all_feature_names.index(
    "Amplitude"
)

datasets = [
    ("Treinamento", X_train),
    ("Ajuste", X_tuning),
    ("Validação", X_val)
]

for name, matrix in datasets:

    # Verificar número de colunas
    assert matrix.shape[1] == len(all_feature_names)

    # Verificar existência de valores inválidos
    assert np.isfinite(matrix).all()

    amplitude = matrix[:, amplitude_index]

    print(f"\nConjunto: {name}")

    print("Dimensões:", matrix.shape)

    print(
        "Percentil 99,5% da amplitude:",
        np.percentile(np.abs(amplitude), 99.5)
    )

    print(
        "Amplitude máxima:",
        np.max(np.abs(amplitude))
    )

    print(
        "Valores finitos:",
        np.isfinite(matrix).all()
    )

print("\nTodas as matrizes foram verificadas!")

#### 6.6. Padronização dos atributos sísmicos

Os atributos sísmicos apresentam diferentes distribuições
e escalas numéricas.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Identificação das colunas da metodologia expandida
feature_indices = [
    all_feature_names.index(name)
    for name in feature_names
]

# Inicialização do padronizador
scaler = StandardScaler()

# Ajustar exclusivamente com o treinamento
X_train_scaled = scaler.fit_transform(
    X_train[:, feature_indices]
)

# Aplicar os mesmos parâmetros ao ajuste
X_tuning_scaled = scaler.transform(
    X_tuning[:, feature_indices]
)

# Aplicar os mesmos parâmetros à validação
X_val_scaled = scaler.transform(
    X_val[:, feature_indices]
)

print("Padronização concluída!")

print("\nDimensões após a padronização:")

print("Treinamento:", X_train_scaled.shape)
print("Ajuste:", X_tuning_scaled.shape)
print("Validação:", X_val_scaled.shape)

In [ ]:
df_scaled_statistics = pd.DataFrame({

    "Atributo": feature_names,

    "Média antes": X_train[
        :, feature_indices
    ].mean(axis=0),

    "Desvio antes": X_train[
        :, feature_indices
    ].std(axis=0),

    "Média depois": X_train_scaled.mean(axis=0),

    "Desvio depois": X_train_scaled.std(axis=0)
})

display(df_scaled_statistics.round(4))

In [ ]:
# Média e desvio padrão do treinamento
scaled_mean = X_train_scaled.mean(axis=0)

scaled_std = X_train_scaled.std(axis=0)

# Identificar atributos não constantes
non_constant = scaler.var_ > 0

# Verificar média aproximadamente zero
assert np.allclose(
    scaled_mean,
    0,
    atol=1e-3
)

# Verificar desvio aproximadamente unitário
assert np.allclose(
    scaled_std[non_constant],
    1,
    atol=1e-3
)

# Verificar valores finitos
assert np.isfinite(X_train_scaled).all()
assert np.isfinite(X_tuning_scaled).all()
assert np.isfinite(X_val_scaled).all()

print("Padronização validada com sucesso!")

print(
    "Atributos constantes:",
    np.array(feature_names)[~non_constant]
)

#### 6.7. Visualização dos efeitos da padronização


In [ ]:
# Gerador para seleção das amostras
local_rng = np.random.default_rng(
    RANDOM_STATE
)

# Selecionar amostras para visualização
plot_indices = local_rng.choice(
    len(X_train),
    size=min(5000, len(X_train)),
    replace=False
)

# Antes da padronização
data_before = X_train[
    plot_indices
][:, feature_indices]

# Depois da padronização
data_after = X_train_scaled[
    plot_indices
]

# Criação dos gráficos
fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 5)
)

# Boxplot antes da padronização
axes[0].boxplot(
    data_before,
    tick_labels=feature_names,
    showfliers=False
)

axes[0].set_title(
    "Antes da padronização"
)

axes[0].set_ylabel(
    "Valor do atributo"
)

# Boxplot depois da padronização
axes[1].boxplot(
    data_after,
    tick_labels=feature_names,
    showfliers=False
)

axes[1].set_title(
    "Depois da padronização"
)

axes[1].set_ylabel(
    "Valor padronizado"
)

# Configurações dos gráficos
for ax in axes:

    ax.tick_params(
        axis="x",
        rotation=45
    )

    ax.grid(
        axis="y",
        alpha=0.3
    )

plt.tight_layout()
plt.show()

### 7. Seleção de Atributos e Análise de Componentes Principais (PCA)

Nesta etapa, é investigada a influência dos atributos sísmicos na qualidade dos agrupamentos

#### 7.1. Análise de correlação entre os atributos

A correlação de Pearson é utilizada para investigar
as relações lineares entre os atributos sísmicos.


In [ ]:
# Matriz de atributos de treinamento
df_correlation = pd.DataFrame(
    X_train,
    columns=all_feature_names
)

# Correlação de Pearson
correlation_matrix = df_correlation.corr()

display(correlation_matrix.round(3))

In [ ]:
fig, ax = plt.subplots(
    figsize=(10, 8)
)

im = ax.imshow(
    correlation_matrix,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

# Identificação dos atributos
ax.set_xticks(
    np.arange(len(all_feature_names))
)

ax.set_yticks(
    np.arange(len(all_feature_names))
)

ax.set_xticklabels(
    all_feature_names,
    rotation=45,
    ha="right"
)

ax.set_yticklabels(
    all_feature_names
)

# Exibir valores da correlação
for i in range(len(all_feature_names)):

    for j in range(len(all_feature_names)):

        value = correlation_matrix.iloc[i, j]

        ax.text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center",
            color="white" if abs(value) > 0.6 else "black"
        )

plt.colorbar(
    im,
    ax=ax,
    label="Correlação de Pearson"
)

ax.set_title(
    "Matriz de Correlação dos Atributos Sísmicos"
)

plt.tight_layout()
plt.show()

In [ ]:
high_correlations = []

for i in range(len(all_feature_names)):

    for j in range(i + 1, len(all_feature_names)):

        correlation = correlation_matrix.iloc[i, j]

        if abs(correlation) >= 0.8:

            high_correlations.append({

                "Atributo 1": all_feature_names[i],

                "Atributo 2": all_feature_names[j],

                "Correlação": correlation
            })

df_high_correlations = pd.DataFrame(
    high_correlations,
    columns=[
        "Atributo 1",
        "Atributo 2",
        "Correlação"
    ]
)

if df_high_correlations.empty:

    print("Nenhuma correlação elevada encontrada.")

else:

    display(df_high_correlations.round(4))

#### 7.2. Definição dos experimentos

São consideradas quatro representações dos dados sísmicos:

- Experimento A: amplitude.
- Experimento B: amplitude, envelope e fase instantânea.
- Experimento C: conjunto expandido de atributos.
- Experimento D: conjunto expandido com aplicação do PCA.

In [ ]:
feature_sets = {

    "A": [
        "Amplitude"
    ],

    "B": original_feature_names,

    "C": feature_names,

    "D": feature_names
}

experiment_names = {

    "A": "Apenas amplitude",

    "B": "Metodologia original",

    "C": "Atributos expandidos",

    "D": "Atributos expandidos + PCA"
}

for experiment, names in feature_sets.items():

    print(
        f"\nExperimento {experiment}:"
    )

    print(experiment_names[experiment])

    print(names)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

experiment_data = {}
experiment_preprocessors = {}

for experiment in ["A", "B", "C", "D"]:

    # Identificar as colunas selecionadas
    indices = [
        all_feature_names.index(name)
        for name in feature_sets[experiment]
    ]

    # Definir as etapas de processamento
    steps = [
        ("scaler", StandardScaler())
    ]

    # Adicionar PCA somente ao experimento D
    if experiment == "D":

        steps.append(
            (
                "pca",
                PCA(
                    n_components=0.95,
                    svd_solver="full"
                )
            )
        )

    preprocessor = Pipeline(steps)

    # Ajustar somente no treinamento
    X_train_exp = preprocessor.fit_transform(
        X_train[:, indices]
    )

    # Transformar o conjunto de ajuste
    X_tuning_exp = preprocessor.transform(
        X_tuning[:, indices]
    )

    # Armazenar os resultados
    experiment_data[experiment] = {

        "train": X_train_exp,

        "tuning": X_tuning_exp,

        "indices": indices
    }

    # Armazenar o pré-processador
    experiment_preprocessors[experiment] = preprocessor

    print(
        f"Experimento {experiment} | "
        f"Treinamento: {X_train_exp.shape} | "
        f"Ajuste: {X_tuning_exp.shape}"
    )

#### 7.3. Análise de Componentes Principais (PCA)

A Análise de Componentes Principais é utilizada para
investigar a possibilidade de reduzir a dimensionalidade
dos atributos sísmicos.

In [ ]:
pca_full = PCA(
    svd_solver="full"
)

pca_full.fit(
    experiment_data["C"]["train"]
)

# Variância explicada individual
explained_variance = (
    pca_full.explained_variance_ratio_
)

# Variância explicada acumulada
cumulative_variance = np.cumsum(
    explained_variance
)

# PCA utilizado no experimento D
pca = experiment_preprocessors[
    "D"
].named_steps["pca"]

n_components = pca.n_components_

print(
    "Número original de dimensões:",
    len(feature_names)
)

print(
    "Número de componentes selecionadas:",
    n_components
)

print(
    "Variância preservada:",
    pca.explained_variance_ratio_.sum()
)

# Tabela de variância explicada
df_pca = pd.DataFrame({

    "Componente": [
        f"PC{i + 1}"
        for i in range(len(explained_variance))
    ],

    "Variância explicada (%)":
        explained_variance * 100,

    "Variância acumulada (%)":
        cumulative_variance * 100
})

display(df_pca.round(4))

In [ ]:
components = np.arange(
    1,
    len(explained_variance) + 1
)

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5)
)

# Variância individual
axes[0].bar(
    components,
    explained_variance * 100,
    color="steelblue"
)

axes[0].set_title(
    "Variância Explicada Individual"
)

axes[0].set_xlabel(
    "Componente principal"
)

axes[0].set_ylabel(
    "Variância explicada (%)"
)

axes[0].set_xticks(
    components
)

# Variância acumulada
axes[1].plot(
    components,
    cumulative_variance * 100,
    marker="o",
    color="darkorange"
)

# Limite de 95%
axes[1].axhline(
    95,
    color="red",
    linestyle="--",
    label="Limite de 95%"
)

# Quantidade de componentes selecionadas
axes[1].axvline(
    n_components,
    color="green",
    linestyle="--",
    label=f"{n_components} componentes"
)

axes[1].set_title(
    "Variância Explicada Acumulada"
)

axes[1].set_xlabel(
    "Número de componentes"
)

axes[1].set_ylabel(
    "Variância acumulada (%)"
)

axes[1].set_xticks(
    components
)

axes[1].legend()

for ax in axes:

    ax.grid(
        alpha=0.3
    )

plt.tight_layout()
plt.show()

In [ ]:
df_pca_components = pd.DataFrame(

    pca.components_.T,

    index=feature_names,

    columns=[
        f"PC{i + 1}"
        for i in range(n_components)
    ]
)

display(
    df_pca_components.round(4)
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(10, 6)
)

im = ax.imshow(

    df_pca_components,

    cmap="coolwarm",

    vmin=-1,

    vmax=1,

    aspect="auto"
)

ax.set_xticks(
    np.arange(n_components)
)

ax.set_xticklabels(
    df_pca_components.columns
)

ax.set_yticks(
    np.arange(len(feature_names))
)

ax.set_yticklabels(
    feature_names
)

for i in range(len(feature_names)):

    for j in range(n_components):

        value = df_pca_components.iloc[i, j]

        ax.text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center",
            color="white" if abs(value) > 0.6 else "black"
        )

plt.colorbar(
    im,
    ax=ax,
    label="Coeficiente da componente"
)

ax.set_title(
    "Coeficientes dos Atributos nas Componentes Principais"
)

plt.tight_layout()
plt.show()

#### 7.4. Comparação das diferentes representações

As diferentes combinações de atributos são avaliadas
utilizando o algoritmo K-means

In [ ]:
from sklearn.cluster import KMeans

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

import time

# Número preliminar de clusters
K_PRELIMINARY = 6

# Gerador aleatório
evaluation_rng = np.random.default_rng(
    RANDOM_STATE
)

# Selecionar as mesmas amostras para todos os experimentos
evaluation_indices = evaluation_rng.choice(

    len(X_tuning),

    size=min(
        SILHOUETTE_SAMPLES,
        len(X_tuning)
    ),

    replace=False
)

# Espaço comum para avaliação
X_reference = experiment_data[
    "C"
]["tuning"][evaluation_indices]

print(
    "Amostras utilizadas na avaliação:",
    len(evaluation_indices)
)

print(
    "Número preliminar de clusters:",
    K_PRELIMINARY
)

print(
    "Dimensões do espaço de referência:",
    X_reference.shape
)

In [ ]:
experiment_results = []
preliminary_models = {}

for experiment in ["A", "B", "C", "D"]:

    # Dados do experimento
    X_train_exp = experiment_data[
        experiment
    ]["train"]

    X_tuning_exp = experiment_data[
        experiment
    ]["tuning"]

    # Modelo preliminar
    kmeans_preliminary = KMeans(

        n_clusters=K_PRELIMINARY,

        random_state=RANDOM_STATE,

        n_init=10,

        max_iter=300
    )

    # Treinamento
    start_time = time.perf_counter()

    kmeans_preliminary.fit(
        X_train_exp
    )

    training_time = (
        time.perf_counter() - start_time
    )

    # Predição sobre o conjunto de ajuste
    labels = kmeans_preliminary.predict(
        X_tuning_exp
    )

    # Seleção das mesmas amostras
    labels_evaluation = labels[
        evaluation_indices
    ]

    X_native = X_tuning_exp[
        evaluation_indices
    ]

    # Quantidade de clusters encontrados
    n_labels = len(
        np.unique(labels_evaluation)
    )

    # Verificar se as métricas podem ser calculadas
    if 2 <= n_labels < len(labels_evaluation):

        # Silhouette no espaço do próprio experimento
        silhouette_native = silhouette_score(
            X_native,
            labels_evaluation
        )

        # Métricas no espaço comum
        silhouette_common = silhouette_score(
            X_reference,
            labels_evaluation
        )

        db = davies_bouldin_score(
            X_reference,
            labels_evaluation
        )

        ch = calinski_harabasz_score(
            X_reference,
            labels_evaluation
        )

    else:

        silhouette_native = np.nan
        silhouette_common = np.nan
        db = np.nan
        ch = np.nan

    # Armazenar o modelo
    preliminary_models[
        experiment
    ] = kmeans_preliminary

    # Armazenar os resultados
    experiment_results.append({

        "Experimento": experiment,

        "Representação": experiment_names[experiment],

        "Dimensões": X_train_exp.shape[1],

        "Clusters encontrados": n_labels,

        "Silhouette": silhouette_common,

        "Silhouette próprio espaço": silhouette_native,

        "Davies-Bouldin": db,

        "Calinski-Harabasz": ch,

        "Tempo (s)": training_time
    })

    print(
        f"Experimento {experiment} concluído!"
    )

df_experiments = pd.DataFrame(
    experiment_results
)

display(
    df_experiments.round(4)
)

In [ ]:
fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 9)
)

experiments = df_experiments[
    "Experimento"
]

metrics = [

    (
        "Silhouette",
        "Coeficiente de Silhouette",
        "steelblue"
    ),

    (
        "Davies-Bouldin",
        "Índice Davies-Bouldin",
        "darkorange"
    ),

    (
        "Calinski-Harabasz",
        "Índice Calinski-Harabasz",
        "seagreen"
    ),

    (
        "Tempo (s)",
        "Tempo de Treinamento (s)",
        "slateblue"
    )
]

for ax, (column, title, color) in zip(
    axes.ravel(),
    metrics
):

    bars = ax.bar(

        experiments,

        df_experiments[column],

        color=color
    )

    ax.set_title(title)

    ax.set_xlabel("Experimento")

    ax.grid(
        axis="y",
        alpha=0.3
    )

    ax.bar_label(
        bars,
        fmt="%.3g",
        padding=3
    )

plt.suptitle(
    "Comparação das Representações dos Atributos Sísmicos",
    fontsize=14
)

plt.tight_layout()
plt.show()

#### 7.5. Seleção da representação dos atributos

A representação dos atributos é selecionada inicialmente
com base no coeficiente de Silhouette calculado no espaço
comum de referência.

Os índices Davies-Bouldin e Calinski-Harabasz, a
dimensionalidade e o custo computacional são considerados
como critérios complementares.

In [ ]:
df_experiments_ranked = (
    df_experiments
    .dropna(subset=["Silhouette"])
    .sort_values(
        by=[
            "Silhouette",
            "Davies-Bouldin"
        ],

        ascending=[
            False,
            True
        ]
    )
    .reset_index(drop=True)
)

if df_experiments_ranked.empty:

    raise ValueError(
        "Nenhum experimento apresentou métricas válidas."
    )

display(
    df_experiments_ranked.round(4)
)

best_experiment = (
    df_experiments_ranked
    .iloc[0]["Experimento"]
)

print(
    "Representação selecionada:",
    best_experiment
)

print(
    "Metodologia:",
    experiment_names[best_experiment]
)

print(
    "Atributos:",
    feature_sets[best_experiment]
)

In [ ]:
original_result = df_experiments.loc[
    df_experiments["Experimento"] == "B"
].iloc[0]

# Resultados da representação selecionada
best_result = df_experiments_ranked.iloc[0]

# Diferença absoluta de Silhouette
silhouette_difference = (

    best_result["Silhouette"]

    - original_result["Silhouette"]
)

print(
    "Silhouette da metodologia original:",
    original_result["Silhouette"]
)

print(
    "Silhouette da representação selecionada:",
    best_result["Silhouette"]
)

print(
    "Diferença absoluta:",
    silhouette_difference
)

print(
    "\nDimensões da metodologia original:",
    original_result["Dimensões"]
)

print(
    "Dimensões da representação selecionada:",
    best_result["Dimensões"]
)

In [ ]:
# Atributos selecionados
selected_features = feature_sets[
    best_experiment
].copy()

# Índices das colunas selecionadas
selected_indices = experiment_data[
    best_experiment
]["indices"]

# Pré-processador selecionado
final_preprocessor = (
    experiment_preprocessors[
        best_experiment
    ]
)

# Matriz definitiva de treinamento
X_train_final = final_preprocessor.transform(
    X_train[:, selected_indices]
)

# Matriz definitiva de ajuste
X_tuning_final = final_preprocessor.transform(
    X_tuning[:, selected_indices]
)

# Matriz definitiva de validação
X_val_final = final_preprocessor.transform(
    X_val[:, selected_indices]
)

# Nomes das dimensões finais
if best_experiment == "D":

    final_feature_names = [

        f"PC{i + 1}"

        for i in range(
            X_train_final.shape[1]
        )
    ]

else:

    final_feature_names = (
        selected_features.copy()
    )

# Verificações
assert np.isfinite(X_train_final).all()
assert np.isfinite(X_tuning_final).all()
assert np.isfinite(X_val_final).all()

assert (
    X_train_final.shape[1]
    == X_tuning_final.shape[1]
    == X_val_final.shape[1]
)

print("Representação:", best_experiment)

print("Atributos:", selected_features)

print("Dimensões finais:", final_feature_names)

print("\nMatrizes definitivas:")

print("Treinamento:", X_train_final.shape)
print("Ajuste:", X_tuning_final.shape)
print("Validação:", X_val_final.shape)

#### 7.6. Visualização dos agrupamentos preliminares


In [ ]:
# PCA exclusivamente para visualização
pca_visual = PCA(
    n_components=2
)

# Ajuste somente com os dados de treinamento
pca_visual.fit(
    experiment_data["C"]["train"]
)

# Projeção das amostras de ajuste
X_visual = pca_visual.transform(
    X_reference
)

print(
    "Dimensões da projeção:",
    X_visual.shape
)

print(
    "Variância preservada na visualização:",
    pca_visual.explained_variance_ratio_.sum()
)

In [ ]:
fig, axes = plt.subplots(
    2, 2,
    figsize=(13, 10),
    sharex=True,
    sharey=True
)

for ax, experiment in zip(

    axes.ravel(),

    ["A", "B", "C", "D"]
):

    # Modelo preliminar
    model = preliminary_models[
        experiment
    ]

    # Dados correspondentes ao experimento
    X_exp = experiment_data[
        experiment
    ]["tuning"][evaluation_indices]

    # Identificação dos clusters
    labels = model.predict(
        X_exp
    )

    # Visualização
    scatter = ax.scatter(

        X_visual[:, 0],

        X_visual[:, 1],

        c=labels,

        cmap="tab10",

        vmin=-0.5,

        vmax=9.5,

        s=5,

        alpha=0.6
    )

    ax.set_title(

        f"Experimento {experiment}\n"

        f"{experiment_names[experiment]}"
    )

    ax.set_xlabel("PC1")

    ax.set_ylabel("PC2")

    ax.grid(
        alpha=0.2
    )

plt.suptitle(
    "Comparação Visual dos Agrupamentos Preliminares",
    fontsize=14
)

plt.tight_layout()
plt.show()

### 8. Agrupamento utilizando K-means

### 9. Agrupamento utilizando Gaussian Mixture Model (GMM)

### 10. Validação Interna dos Modelos

### 11. Generalização entre Bacias Sedimentares

### 12. Análise de Desempenho Computacional

### 13. Resultados e Discussões